# Model Training and Evaluation — Concepts

Slides **5–17**, `Slides_MSc_Data_Science_Week_06_EN.pdf`.

Read the idea, the formula and the small example in that order. The demonstration code is provided.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

## 1. What does machine learning do?

Machine learning learns a rule from examples. A row is an observation, a feature $x$ an input, and the target $y$ the answer. A prediction is $\hat y$ ("y hat"). Engine size can be an input for predicting fuel consumption.

| Task | Data | Question | Example |
|---|---|---|---|
| Regression | inputs and numerical target | How much? | monthly rent |
| Classification | inputs and class label | Which class? | survived: yes/no |
| Unsupervised learning | inputs only | Which groups or patterns? | groups of similar cars |
| Reinforcement learning | actions and rewards | Which action helps? | playing a game |

Reinforcement learning is outside this course. The goal is good predictions for **new observations**.

**History (slide 9):** Bayes (1763), least squares (1805), Turing (1936), perceptron (1957), nearest neighbours (1967), CART (1984), boosting (1990), SVM (1995), random forests (2001), Transformers (2017).

## 2. Loss and error measures

The error is $e=y-\hat y$. Squared loss is $e^2$; absolute loss is $|e|$. Squaring gives large errors more weight.
$$\mathrm{MSE}=\frac1n\sum(y_i-\hat y_i)^2,\quad \mathrm{RMSE}=\sqrt{\mathrm{MSE}},\quad \mathrm{MAE}=\frac1n\sum|y_i-\hat y_i|.$$
$n$ is the number of observations; $\sum$ means "add up". RMSE and MAE have the units of the target. For errors 1 and 3: MSE = 5, RMSE ≈ 2.24 and MAE = 2.

For classification, **0–1 loss** is 1 for a wrong label and 0 for a correct one. **Log-loss** penalises confidently wrong predictions. **Hinge loss** also asks for some distance from the boundary. For labels −1/+1 and score $s$, the margin is $m=ys$:
$$L_{\log}=\log(1+e^{-m}),\qquad L_{\mathrm{hinge}}=\max(0,1-m).$$
Recognise the ideas; no derivation is required.

In [ ]:
errors = np.array([1, 3])
print('MSE:', np.mean(errors ** 2))
print('RMSE:', np.sqrt(np.mean(errors ** 2)))
print('MAE:', np.mean(np.abs(errors)))
e = np.linspace(-3, 3, 100)
plt.plot(e, e ** 2, label='Squared loss')
plt.plot(e, np.abs(e), label='Absolute loss')
plt.xlabel('Error')
plt.ylabel('Loss')
plt.legend()
plt.show()

## 3. Loss, objective and metric

A **loss** concerns one prediction. The **training objective** combines losses, sometimes with a penalty. The **evaluation metric** measures how the fitted model works on new data.

| Model | Training objective | Evaluation metric |
|---|---|---|
| Linear regression | sum of squared errors | RMSE, R² |
| Logistic regression | log-loss | accuracy, recall, AUC |
| Tree | decrease SSE or Gini/entropy per split | RMSE or classification metrics |
| SVM | hinge loss plus a penalty | accuracy, AUC |
| Gradient boosting | squared loss or log-loss | depends on the task |

Training searches for a rule with small average loss:
$$\hat f=\arg\min_f\frac1n\sum L(y_i,f(x_i)).$$
$f$ is a candidate rule; $\arg\min$ means "choose the rule with the smallest value". The objective and reported metric may differ.

## 4. Train/test split and cross-validation

Fit the model on about 80% of the rows and reserve 20% for a final test. Training scores are often optimistic because the model has already seen these rows.

For **5-fold cross-validation (CV)**, split the training set into five parts. Fit on four and validate on the remaining one; repeat five times:
$$\mathrm{CV\ error}=\frac{\mathrm{error}_1+\cdots+\mathrm{error}_5}{5}.$$
Use CV for model and parameter choices. Use test data after the choices are fixed. Learn preprocessing from training rows; in CV, put preprocessing in a pipeline so that each fold learns it separately.

## 5. Underfitting, overfitting and bias–variance

**Underfitting:** a model is too simple and misses the pattern. **Overfitting:** it learns random details as well; training error is low but validation error is higher.

**Bias** is systematic error from overly simple assumptions. **Variance** means predictions change a lot with another training sample. Some noise cannot be predicted:
$$\mathrm{expected\ squared\ test\ error}=\mathrm{bias}^2+\mathrm{variance}+\mathrm{noise}.$$
More data, simpler models and averaging can reduce variance. Better features or more flexible models can reduce bias. We want a useful balance.

The provided plot compares three tree sizes. Read the curves; you do not need to write the plotting loop.

In [ ]:
rng = np.random.default_rng(42)
x = np.linspace(0, 6, 40)
y = np.sin(x) + rng.normal(0, 0.2, len(x))
grid = np.linspace(0, 6, 200).reshape(-1, 1)
plt.scatter(x, y, color='black', label='Observations')
for depth in [1, 3, 15]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree.fit(x.reshape(-1, 1), y)
    plt.plot(grid[:, 0], tree.predict(grid), label=f'Depth {depth}')
plt.xlabel('Input x')
plt.ylabel('Target y')
plt.legend()
plt.show()

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))